# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Greemines/Flyrank-Notebook-1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

##**Finding 1** — Logistic Regression growth classification

The paper observed 71% holdout accuracy from a Logistic Regression model for separating growing and declining pages. My methodology question is: How was the growing-versus-declining label constructed, and does the 80/20 validation design support this result for unseen clients or future data? A grouped or time-aware validation design could provide additional evidence about whether the measured performance generalizes beyond the original split. This finding should therefore be treated as directional evidence and potential decision-support, rather than as a guarantee of performance in new data.

##**Finding 2**  — Random Forest feature importance

The paper measured Average Position and Impressions as the strongest Random Forest features when predicting Health Score. My methodology question is: Because Health Score is constructed partly from impressions, position, CTR, and scroll depth, how much of the measured feature importance reflects the model reconstructing inputs already present in the target? This makes the result useful as a directional, descriptive finding, but it should not be interpreted as evidence that these variables independently cause changes in Health Score. It is better treated as decision-support for further investigation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

##**Validation design**

I first measure the Logistic Regression model using a conventional random 80/20 split as a less strict reference point. I then re-run the same model using a grouped split by client_id, so no client appears in both training and test data. The model, target, features, and evaluation metrics remain the same in both cases. This provides a direct before/after comparison of the validation design.

The grouped result is the more conservative estimate for this task because it measures performance on clients that were not present during training. The comparison is treated as measured and directional evidence about generalization, not as a guarantee of future performance.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


df = pd.read_csv("/content/content_refresh_anonymized.csv")


features = [
    "impressions_90d",
    "ctr",
    "avg_position"
]

df_model = df[
    features + ["client_id", "trend_direction", "content_id"]
].copy()


df_model["target"] = (
    df_model["trend_direction"] == "down"
).astype(int)




def run_model(train_df, test_df):

    X_train = train_df[features].copy()
    X_test = test_df[features].copy()

    y_train = train_df["target"]
    y_test = test_df["target"]


    X_train["impressions_90d"] = np.log1p(
        X_train["impressions_90d"]
    )

    X_test["impressions_90d"] = np.log1p(
        X_test["impressions_90d"]
    )

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]

    results = test_df[
        [
            "content_id",
            "client_id",
            "target"
        ]
    ].copy()

    results["model_score"] = probabilities

    ranking = results.sort_values(
        "model_score",
        ascending=False
    ).reset_index(drop=True)

    precision_20 = ranking.head(20)["target"].mean()
    precision_50 = ranking.head(50)["target"].mean()

    return ranking, precision_20, precision_50


train_random, test_random = train_test_split(
    df_model,
    test_size=0.20,
    random_state=42,
    stratify=df_model["target"]
)

random_ranking, random_p20, random_p50 = run_model(
    train_random,
    test_random
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df_model[features],
        df_model["target"],
        groups=df_model["client_id"]
    )
)

train_grouped = df_model.iloc[train_idx].copy()
test_grouped = df_model.iloc[test_idx].copy()

grouped_ranking, grouped_p20, grouped_p50 = run_model(
    train_grouped,
    test_grouped
)



random_clients_train = set(train_random["client_id"])
random_clients_test = set(test_random["client_id"])

grouped_clients_train = set(train_grouped["client_id"])
grouped_clients_test = set(test_grouped["client_id"])


print("RANDOM SPLIT")
print("Training rows:", len(train_random))
print("Test rows:", len(test_random))
print(
    "Client overlap:",
    len(random_clients_train & random_clients_test)
)

print("\nGROUPED SPLIT")
print("Training rows:", len(train_grouped))
print("Test rows:", len(test_grouped))
print(
    "Training clients:",
    len(grouped_clients_train)
)
print(
    "Test clients:",
    len(grouped_clients_test)
)
print(
    "Client overlap:",
    len(grouped_clients_train & grouped_clients_test)
)

comparison = pd.DataFrame({
    "Validation": [
        "Random 80/20",
        "Grouped by client"
    ],
    "Precision@20": [
        random_p20 * 100,
        grouped_p20 * 100
    ],
    "Precision@50": [
        random_p50 * 100,
        grouped_p50 * 100
    ]
})

comparison["Precision@20"] = comparison[
    "Precision@20"
].round(2)

comparison["Precision@50"] = comparison[
    "Precision@50"
].round(2)

print("\nBefore / After Validation Comparison:")
display(comparison)

RANDOM SPLIT
Training rows: 24000
Test rows: 6000
Client overlap: 31

GROUPED SPLIT
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0

Before / After Validation Comparison:


,Validation,Precision@20,Precision@50
0,Random 80/20,60.0,52.0
1,Grouped by client,35.0,44.0


##**Before / after interpretation**

The random 80/20 split measured Precision@20 of 60% and Precision@50 of 52%. However, pages from 31 clients appeared in both the training and test sets, so this split does not measure generalization to unseen clients.

Under the grouped-by-client split, Precision@20 was 35% and Precision@50 was 44%, with zero client overlap between training and test data. The measured performance was therefore lower under the stricter validation design. This provides directional evidence that the random split may give a more optimistic estimate of performance for unseen clients.

For this task, the grouped result is the more useful decision-support measurement because the test set contains clients that were not used during training. These results do not establish how the model will perform on future data.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

##**Leakage audit**

My final model uses impressions_90d, ctr, and avg_position. I check these features against the target definition and also inspect the dataset for fields that may directly encode the outcome or use future-window information. The target is trend_direction, where down is converted to 1 and all other directions to 0.

I do not use trend_direction, trend_pct, or other target-derived fields as model features. The audit is intended to identify potential leakage before interpreting the model as decision-support. A feature is treated as potentially unsafe if it directly contains the target, is derived from the target, or represents information from a future observation window.

In [2]:
import pandas as pd

df = pd.read_csv(
    "/content/content_refresh_anonymized.csv"
)

features = [
    "impressions_90d",
    "ctr",
    "avg_position"
]

target = "trend_direction"

print("Final model features:")
print(features)

print("\nTarget:")
print(target)

direct_target_leakage = [
    col for col in features
    if col == target
]

print("\nDirect target leakage:")
print(direct_target_leakage)

assert len(direct_target_leakage) == 0, \
    "Target is being used directly as a feature."


target_derived = [
    "trend_direction",
    "trend_pct"
]

used_target_derived = [
    col for col in features
    if col in target_derived
]

print("\nTarget-derived fields used as features:")
print(used_target_derived)

assert len(used_target_derived) == 0, \
    "A target-derived field is being used as a feature."


future_or_target_like = [
    col for col in df.columns
    if any(
        word in col.lower()
        for word in [
            "future",
            "trend",
            "target",
            "label",
            "next"
        ]
    )
]

print("\nPotential future/target-like columns:")
print(future_or_target_like)

print("\nFinal feature columns found in dataset:")

for col in features:
    print(
        f"{col}:",
        "FOUND" if col in df.columns else "MISSING"
    )

assert all(
    col in df.columns for col in features
), "A model feature is missing from the dataset."


print("\nFinal feature set:")
display(
    df[features].head()
)

print("\nLeakage audit summary:")
print("Direct target leakage: PASS")
print("Target-derived feature leakage: PASS")
print("Final feature columns available: PASS")
print("Final features:", features)

Final model features:
['impressions_90d', 'ctr', 'avg_position']

Target:
trend_direction

Direct target leakage:
[]

Target-derived fields used as features:
[]

Potential future/target-like columns:
['trend_direction', 'trend_pct']

Final feature columns found in dataset:
impressions_90d: FOUND
ctr: FOUND
avg_position: FOUND

Final feature set:


,impressions_90d,ctr,avg_position
0,3803,0.76,10.6
1,15320,0.05,20.3
2,12581,0.09,36.5
3,11751,0.49,6.2
4,19140,0.13,44.0



Leakage audit summary:
Direct target leakage: PASS
Target-derived feature leakage: PASS
Final feature columns available: PASS
Final features: ['impressions_90d', 'ctr', 'avg_position']


##**Leakage result**
The audit found no identified direct target leakage in the final feature set. The model uses only impressions_90d, ctr, and avg_position; neither trend_direction nor trend_pct is used as a feature. trend_direction is used to construct the binary target and evaluate predictions, while trend_pct remains excluded from the model features.

The audit therefore found no identified direct target leakage in the final feature set. This supports using the model as directional decision-support, but does not establish that every possible form of leakage has been ruled out.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Under the measured grouped-by-client validation, the Logistic Regression model achieved 35% Precision@20 and 44% Precision@50. In the Week-5 comparison, it measured 35% Precision@20 versus 30% for the baseline. The results provide directional evidence of a modest ranking improvement at Precision@20, but the improvement is limited and performance was lower under the stricter grouped validation than under the random split. The model should therefore be treated as decision-support for prioritizing pages for review rather than as a definitive predictor of future decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.